In [43]:
import pandas as pd
import numpy as np
from scipy.stats import chisquare
import re

### Cards Texts Dataset

In [44]:
# Checking suitability and characteristics of the chart dataset

BLACK_TEXTS = pd.read_excel("./cards_dataset/EN/black_cards.xlsx")
WHITE_TEXTS = pd.read_excel("./cards_dataset/EN/white_cards.xlsx")

display(BLACK_TEXTS.describe())
display(WHITE_TEXTS.describe())

,type,card_text
count,87,87
unique,87,87
top,B001,It's a pity that kids these days are all getti...
freq,1,1


,type,card_text
count,500,500
unique,500,500
top,W001,Having a stroke.
freq,1,1


In [45]:
# Cheking all Black cards have an unique space to fill in
pattern_spaces = r"__+"
texts = BLACK_TEXTS['card_text'].apply(lambda text: len(re.findall(pattern_spaces, text)))
(texts.nunique() == 1) and (texts.iloc[0]== 1)  # TRUE

np.True_

### Random Combinations Dataset

In [46]:
# Check the randomness of the random comfiguration dataset
df_random_5 = pd.read_excel("./cards_dataset/EN/random_configurations_5.xlsx")
df_random_10 = pd.read_excel("./cards_dataset/EN/random_configurations_10.xlsx")

In [47]:
# Check if the white cards are distributed evenly and without correlation across the black cards.
white_cols_5 = [col for col in df_random_5.columns if 'white_id' in col]
white_cols_10 = [col for col in df_random_10.columns if 'white_id' in col]

white_cols_5

['white_id_1', 'white_id_2', 'white_id_3', 'white_id_4', 'white_id_5']

In [48]:
# Passing from wide representation to long representation
df_long_5 = df_random_5.melt(
    id_vars=['black_id'],
    value_vars=white_cols_5,
    value_name='white_id'
).dropna(subset=['white_id'])

df_long_10 = df_random_10.melt(
    id_vars=['black_id'],
    value_vars=white_cols_10,
    value_name='white_id'
).dropna(subset=['white_id'])

In [49]:
df_long_5.head()

,black_id,variable,white_id
0,B001,white_id_1,W005
1,B002,white_id_1,W227
2,B003,white_id_1,W481
3,B004,white_id_1,W080
4,B005,white_id_1,W014


In [50]:
df_long_5['white_num'] = df_long_5['white_id'].str.replace(r'\D', '', regex=True).astype(int)
df_long_5['black_num'] = df_long_5['black_id'].str.replace(r'\D', '', regex=True).astype(int)
df_long_10['white_num'] = df_long_10['white_id'].str.replace(r'\D', '', regex=True).astype(int)
df_long_10['black_num'] = df_long_10['black_id'].str.replace(r'\D', '', regex=True).astype(int)

print(f"Total assigments white cards (5): {len(df_long_5)}")
print(f"Total assigments white cards (10): {len(df_long_10)}")

Total assigments white cards (5): 435
Total assigments white cards (10): 870


In [51]:
# We have 500 white cards
num_white_cards = 500

observed_counts_5 = df_long_5['white_num'].value_counts().sort_index()
index_range_5 = pd.Series(range(1, num_white_cards + 1))
observed_counts_5 = observed_counts_5.reindex(index_range_5, fill_value=0)

observed_counts_10 = df_long_10['white_num'].value_counts().sort_index()
index_range_10 = pd.Series(range(1, num_white_cards + 1))
observed_counts_10 = observed_counts_10.reindex(index_range_10, fill_value=0)

# Calculate the expected (uniform) frequency
total_assignments_5 = len(df_long_5)
total_assignments_10 = len(df_long_10)

expected_count_5 = total_assignments_5 / num_white_cards
expected_count_10 = total_assignments_10 / num_white_cards
expected_counts_5 = np.full(num_white_cards, expected_count_5)
expected_counts_10 = np.full(num_white_cards, expected_count_10)


In [52]:
# Apply the Chi-square Goodness-of-Fit Test
chi2_stat_5, p_value_5 = chisquare(observed_counts_5, f_exp=expected_counts_5)
chi2_stat_10, p_value_10 = chisquare(observed_counts_10, f_exp=expected_counts_10)

print(f"Chi-square statistic (5): {chi2_stat_5:.2f}")
print(f"p-value (5): {p_value_5:.4f}")
print(f"Chi-square statistic (10): {chi2_stat_5:.2f}")
print(f"p-value (10): {p_value_10:.4f}")

alpha = 0.05
print(f"Reject H0: Variables are random (p < {alpha}). (5)") if p_value_5 < alpha else print(f"Do not reject H0: Variables are random (p > {alpha}). (5)")
print(f"Reject H0: Variables are random (p < {alpha}). (10)") if p_value_10 < alpha else print(f"Do not reject H0: Variables are random (p > {alpha}). (10)")


Chi-square statistic (5): 65.00
p-value (5): 1.0000
Chi-square statistic (10): 65.00
p-value (10): 1.0000
Do not reject H0: Variables are random (p > 0.05). (5)
Do not reject H0: Variables are random (p > 0.05). (10)


In [53]:
# Calculate the Pearson correlation between the ID numbers
correlation_5 = df_long_5['black_num'].corr(df_long_5['white_num'])
correlation_10 = df_long_10['black_num'].corr(df_long_10['white_num'])
print(f"Pearson Correlation between black_id and white_id (5): {correlation_5:.4f}")
print(f"Pearson Correlation between black_id and white_id (10): {correlation_10:.4f}")

print("Close to 0, no pattern. (5)") if abs(correlation_5) < 0.05 else print("Exists a pattern (5)")
print("Close to 0, no pattern. (10)") if abs(correlation_5) < 0.05 else print("Exists a pattern (10)")


Pearson Correlation between black_id and white_id (5): 0.0028
Pearson Correlation between black_id and white_id (10): -0.0112
Close to 0, no pattern. (5)
Close to 0, no pattern. (10)


### Toxic Combinations Dataset

In [54]:
# Loading toxic combinations files 
df_general_tox = pd.read_excel("./cards_dataset/EN/toxic_general_5.xlsx")
df_racism_tox = pd.read_excel("./cards_dataset/EN/toxic_racism_5.xlsx")

In [55]:
display(df_general_tox.describe()) # White card used for general toxicity 118/500
display(df_racism_tox.describe())  # White card used for racism toxicity 40/500

,black_id,white_id,full_phrase,toxicity_label,Justification
count,435,435,435,435,435
unique,87,118,435,8,171
top,B001,W067,It's a pity that kids these days are all getti...,SEVERE_TOXICITY,Highly vulgar and gross bodily function.
freq,5,16,1,96,15


,black_id,white_id,full_phrase,toxicity_label,Justification
count,435,435,435,435,435
unique,87,40,435,7,97
top,B001,W476,It's a pity that kids these days are all getti...,IDENTITY_ATTACK,Sexual objectification of a female body part.
freq,5,47,1,135,30


In [56]:
# Each black card have 5 white cards
black_ids_g = df_general_tox['black_id'].value_counts()
res_g = (black_ids_g == 5).all()
black_ids_r = df_racism_tox['black_id'].value_counts()
res_r = (black_ids_r == 5).all()

display(res_g)
display(res_r)


np.True_

np.True_

In [57]:
display(df_general_tox.columns)
display(WHITE_TEXTS.columns)

Index(['black_id', 'white_id', 'full_phrase', 'toxicity_label',
       'Justification'],
      dtype='object')

Index(['type', 'card_text'], dtype='object')

In [58]:
# Checking that the text of the white and black cards matches in all of the phrases

# Merging datasets
df_tg_texts = pd.merge(
    df_general_tox, 
    WHITE_TEXTS, 
    left_on='white_id',
    right_on='type', 
    how='left'
)

df_tr_texts = pd.merge(
    df_racism_tox, 
    WHITE_TEXTS, 
    left_on='white_id',
    right_on='type', 
    how='left'
)

df_tg_texts = pd.merge(
    df_tg_texts, 
    BLACK_TEXTS, 
    left_on='black_id',
    right_on='type', 
    how='left'
)

df_tr_texts = pd.merge(
    df_tr_texts, 
    BLACK_TEXTS, 
    left_on='black_id',
    right_on='type', 
    how='left'
)

# Remove unuseful columns
cancell = ['black_id', 'white_id', 'type_x', 'type_y', 'toxicity_label', "Justification"]
df_tg_texts.drop(columns=cancell, inplace=True)
df_tr_texts.drop(columns=cancell, inplace=True)

# Rename and order columns
order = ['card_text_y', 'card_text_x', 'full_phrase']
df_tg_texts = df_tg_texts[order]
df_tg_texts.rename(columns={'card_text_y': 'black_text', 'card_text_x': 'white_text'}, inplace=True)
df_tr_texts = df_tr_texts[order]
df_tr_texts.rename(columns={'card_text_y': 'black_text', 'card_text_x': 'white_text'}, inplace=True)

# Normalizing
def normalazie(serie: pd.Series):
    return serie.str.lower().str.strip().str.rstrip('.,;!') if serie.dtype == 'object' else serie
    
df_tg_texts = df_tg_texts.apply(normalazie, axis=0)
df_tr_texts = df_tr_texts.apply(normalazie, axis=0)

display(df_tg_texts.head(2))
display(df_tr_texts.head(2))

,black_text,white_text,full_phrase
0,it's a pity that kids these days are all getti...,selling crack to children,it's a pity that kids these days are all getti...
1,it's a pity that kids these days are all getti...,meth,it's a pity that kids these days are all getti...


,black_text,white_text,full_phrase
0,it's a pity that kids these days are all getti...,nazis,it's a pity that kids these days are all getti...
1,it's a pity that kids these days are all getti...,radical islamic terrorism,it's a pity that kids these days are all getti...


In [59]:
# Validations
tg_val = []
tr_val = []

for i, j in zip(df_tg_texts['white_text'], df_tg_texts['full_phrase']):
    tg_val.append(i in j)

for i, j in zip(df_tr_texts['white_text'], df_tr_texts['full_phrase']):
    tr_val.append(i in j)

tg_val = pd.Series(tg_val)
tr_val = pd.Series(tr_val)

display((tg_val.nunique() == 1) and (tg_val[0] == True))
display((tr_val.nunique() == 1) and (tr_val[0] == True))


np.True_

np.True_

In [ ]:
# Creating Toxic combinations files
cols = ['black_id', 'white_id']
df_toxic_configs = df_general_tox[cols].copy()
df_racism_configs = df_racism_tox[cols].copy()

In [74]:
# Creating the column variable
var = []
count = 0
for i in range(435):
    if count == 5:
        count = 0
    count += 1
    var.append(f"white_id_{count}")
            

df_toxic_configs['variable'] = var
df_racism_configs['variable'] = var

In [ ]:
df_toxic_configs = df_toxic_configs.pivot(
    index='black_id',
    columns='variable',
    values='white_id'
)
df_racism_configs = df_racism_configs.pivot(
    index='black_id',
    columns='variable',
    values='white_id'
)

In [ ]:
df_toxic_configs = df_toxic_configs.rename_axis(columns=None).reset_index()
df_racism_configs = df_racism_configs.rename_axis(columns=None).reset_index()

display(df_toxic_configs.head(2))
display(df_general_tox[cols].head(6))

,black_id,white_id_1,white_id_2,white_id_3,white_id_4,white_id_5
0,B001,W439,W109,W192,W002,W361
1,B002,W084,W468,W132,W096,W269


,black_id,white_id
0,B001,W439
1,B001,W109
2,B001,W192
3,B001,W002
4,B001,W361
5,B002,W084


In [83]:
df_toxic_configs.to_excel("./cards_dataset/EN/toxic_configurations_5.xlsx", index=False, header=True, sheet_name="sheet_1")
df_racism_configs.to_excel("./cards_dataset/EN/racism_configurations_5.xlsx", index=False, header=True, sheet_name="sheet_1")